In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
customer_df = spark.table("olist_eccom_catalog.bronze.customer_bronze")
orders_df = spark.table("olist_eccom_catalog.bronze.orders_bronze")
payments_df = spark.table("olist_eccom_catalog.bronze.payments_bronze")
items_df = spark.table("olist_eccom_catalog.bronze.order_items_bronze")
products_df = spark.table("olist_eccom_catalog.bronze.products_bronze")
reviews_df = spark.table("olist_eccom_catalog.bronze.reviews_bronze")
sellers_df = spark.table("olist_eccom_catalog.bronze.sellers_bronze")


### Params

In [0]:
variable_name ="customer_bronze"

Trim

In [0]:
 
def trim_data(df):
    get_dtypes = dict(df.dtypes)
    print(get_dtypes)
    for c, t in get_dtypes.items():
        print(c, t)
        if t == "string":
            df = df.select([trim(col(c)).alias(c) for c in df.columns])
        else:
            col(c)

    return df

customer_df = trim_data(customer_df)
orders_df = trim_data(orders_df)
payments_df = trim_data(payments_df)
items_df = trim_data(items_df)
products_df = trim_data(products_df)
reviews_df = trim_data(reviews_df)
sellers_df = trim_data(sellers_df)

cast date string to timesytamp


In [0]:
# casting customer_df date string to timestamp

# orders_df.display()

orders_df = orders_df.withColumn("order_purchase_timestamp", to_timestamp(orders_df.order_purchase_timestamp)).withColumn("order_approved_at", to_timestamp(orders_df.order_approved_at)).withColumn("order_delivered_carrier_date", to_timestamp(orders_df.order_delivered_carrier_date)).withColumn("order_estimated_delivery_date", to_timestamp(orders_df.order_estimated_delivery_date)).withColumn("order_delivered_customer_date", to_timestamp(orders_df.order_delivered_customer_date))

orders_df.display()


In [0]:
items_df = items_df.withColumn("shipping_limit_date", to_timestamp(items_df.shipping_limit_date))

items_df.display()

In [0]:
reviews_df = reviews_df.withColumn("review_creation_date", try_to_timestamp(col("review_creation_date"))).withColumn("review_answer_timestamp", try_to_timestamp(col("review_answer_timestamp"))).filter(col("review_creation_date").isNotNull()).filter(col("review_answer_timestamp").isNotNull())
reviews_df.display()

remove nulls

In [0]:
products_df = products_df.filter((col("product_category_name").isNotNull()) & (col("product_description_lenght").isNotNull()) & (col("product_name_lenght").isNotNull()))
products_df.display()


In [0]:
# no nulls in sellers

# sellers_df_nulls = sellers_df.filter((col("seller_city").isNull()) | (col("seller_state").isNull()) | (col("seller_zip_code_prefix").isNull()))
# sellers_df_nulls.display() 




## no nulls in payments_df

# payments_df_nulls = payments_df.filter((col("payment_sequential").isNull()) | (col("payment_installments").isNull()) | (col("payment_value").isNull()) | (col("payment_type").isNull()))
# payments_df_nulls.display()

## nulls in orders_df but in dates
# orders_df_nulls = orders_df.filter((col("order_purchase_timestamp").isNull()) | (col("order_approved_at").isNull()) | (col("order_delivered_carrier_date").isNull()) | (col("order_estimated_delivery_date").isNull()) | (col("order_delivered_customer_date").isNull()))
# orders_df_nulls.display()


## no nulls in items df
# items_df_nulls = items_df.filter((col("shipping_limit_date").isNull()) | (col("price").isNull()) | (col("freight_value").isNull()))
# items_df_nulls.display()


## no nulls in customer df
# customer_df_nulls = customer_df.filter((col("customer_city").isNull()) | (col("customer_state").isNull()) | (col("customer_zip_code_prefix").isNull()))
# customer_df_nulls.display()

In [0]:
invalid_items = orders_df.join(items_df, orders_df.order_id == items_df.order_id, "leftanti")
invalid_items.display()


In [0]:
## no invalid customers

invalid_customers_df = orders_df.join(customer_df, orders_df.customer_id == customer_df.customer_id, "leftanti")
invalid_customers_df.display()



In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS olist_eccom_catalog.silver

In [0]:
## Write these table to silver

invalid_items.write.format("delta").mode("overwrite").saveAsTable("olist_eccom_catalog.silver.invalid_order_items")
customer_df.write.format("delta").mode("overwrite").saveAsTable("olist_eccom_catalog.silver.customers")
orders_df.write.format("delta").saveAsTable("olist_eccom_catalog.silver.orders")
items_df.write.format("delta").saveAsTable("olist_eccom_catalog.silver.items")
payments_df.write.format("delta").saveAsTable("olist_eccom_catalog.silver.payments")
products_df.write.format("delta").saveAsTable("olist_eccom_catalog.silver.products")
reviews_df.write.format("delta").saveAsTable("olist_eccom_catalog.silver.reviews")
sellers_df.write.format("delta").saveAsTable("olist_eccom_catalog.silver.sellers")
